In [ ]:
# ─────────────────────────────────────────
# CELL 1 — MOUNT DRIVE  (v8-CLF5)
# ─────────────────────────────────────────
# VOICE_INDEX is gone: there is no reference voice bank in this build.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = "/content/drive/MyDrive/dub_pipeline_assets"
OUTPUT_DIR = f"{DRIVE_BASE}/outputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/refs", exist_ok=True)


In [ ]:
# ─────────────────────────────────────────
# CELL 2 — INSTALL DEPENDENCIES  (v8, Cross-Lingual F5-TTS)
# (run once per session; then RESTART RUNTIME)
# ─────────────────────────────────────────
# coqui-tts is GONE from this branch — f5-tts and coqui-tts fight over
# transformers, and we only need one of them. f5-tts + whisperx is the
# combination v6.1 already ran successfully, so this is a known-good stack.
#
# Cross-Lingual F5-TTS = the base F5-TTS code + a different checkpoint +
# two extra modules from the authors' Space (fetched in CELL 2c).
import subprocess, sys

def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                 "coqui-tts", "TTS"])          # remove the XTTS branch's stack

pip("demucs")
pip("ffmpeg-python")
pip("librosa", "soundfile")
pip("git+https://github.com/m-bain/whisperx.git")
pip("pyannote.audio>=4.0,<5")
pip("sentencepiece", "sacremoses")
pip("pyloudnorm", "scipy")
pip("pyphen", "cached_path")                   # CLF5 syllable counter + HF cache
pip("f5-tts")                                  # last — owns transformers

pip("-U", "numpy>=2.1,<2.5", "numba>=0.66")    # numba caps numpy at <2.5

import numpy, transformers, torch
print(f"numpy {numpy.__version__} | transformers {transformers.__version__} "
      f"| torch {torch.__version__}")
print("✅ RESTART THE RUNTIME NOW, then run CELL 2b.")


In [ ]:
# ─────────────────────────────────────────
# CELL 2b — DEPENDENCY SMOKE TEST  (after restarting the runtime)
# ─────────────────────────────────────────
import importlib
ok = True
for mod in ["whisperx", "transformers", "pyannote.audio", "f5_tts", "pyphen", "demucs"]:
    try:
        m = importlib.import_module(mod)
        print(f"  ✅ {mod:18s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        ok = False
        print(f"  ❌ {mod:18s} {type(e).__name__}: {e}")
print("\n✅ All stacks import together." if ok else
      "\nDo NOT downgrade numpy below 2.1 or transformers below 4.48 — "
      "whisperx needs both.")


In [ ]:
# ─────────────────────────────────────────
# CELL 2c — FETCH CROSS-LINGUAL F5-TTS MODULES
# ─────────────────────────────────────────
# The CLF5 inference path is NOT in the pip package. Two modules live only in
# the authors' Space (chenxie95/Cross-Lingual_F5-TTS_Space):
#   module_clf5.py       — SpeedPredictor architecture
#   utils_clf5_space.py  — infer_process_clf5, the transcript-free inference
# Checkpoints come from QingyuLiu1/Cross-Lingual_F5-TTS (Apache-2.0).
import sys, os
from huggingface_hub import hf_hub_download

CLF5_DIR = "/content/clf5"
os.makedirs(CLF5_DIR, exist_ok=True)
for f in ["module_clf5.py", "utils_clf5_space.py"]:
    p = hf_hub_download(repo_id="chenxie95/Cross-Lingual_F5-TTS_Space",
                        filename=f, repo_type="space", local_dir=CLF5_DIR)
    print("  ✓", p)
if CLF5_DIR not in sys.path:
    sys.path.insert(0, CLF5_DIR)
print("✅ CLF5 modules ready")


In [ ]:
# ─────────────────────────────────────────
# CELL 3 — CONFIG & IMPORTS  (v8, Cross-Lingual F5-TTS)
# ─────────────────────────────────────────
# Deleted vs v6.1: every pitch/gender/age key (pitch_*, male_max_hz,
# female_min_hz, child_*, profile_*, speaker_merge_hz, rank_fallback_gap_hz,
# speaker_gender_override). Nothing downstream reads them any more.
import json, shutil, tempfile, time, gc, os
from pathlib import Path
from typing import Optional
import numpy as np
import torch
import librosa
import soundfile as sf

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")   # Colab sidebar → key icon → secret "HF_TOKEN"

os.environ["PYTHONHASHSEED"] = "0"    # demucs child processes crash without it

CFG = {
    # Pipeline
    "whisperx_model":        "large-v3",
    "nllb_model":            "facebook/nllb-200-distilled-1.3B",
    # Source language. None = auto-detect per video (any language → English).
    # Set an explicit Whisper code ("hi","te","ta","kn","ml","bn","mr","gu"…)
    # to force it — worth doing when auto-detect is wrong on a noisy clip.
    "asr_lang":              None,
    "src_lang_fallback":     "hin_Deva",   # used only if detection is unmapped
    "tgt_lang":              "eng_Latn",
    "no_repeat_ngram_size":  3,
    "repetition_penalty":    1.15,
    "max_word_run":          2,

    # Separation
    "demucs_model":          "htdemucs",   # → "htdemucs_ft" if clone refs sound
                                           # muddy on the music-heavy clips.
                                           # ~4x slower, better vocal isolation.
    "demucs_device":         "cuda",
    "demucs_segment":        7,

    # Diarization
    "diar_min_speakers":     1,
    "diar_max_speakers":     6,

    # ── Source-cloning reference selection (replaces the 36-clip voice bank)
    "ref_target_sec":        8.0,    # CLF5 budgets 22 s TOTAL (prompt +
                                    # generated). A 20 s prompt leaves ~2 s and
                                    # the text gets chunked to pieces. Keep short.
    "ref_min_seg_sec":       1.5,    # ignore shorter segments as ref candidates
    "ref_max_seg_sec":       8.0,    # cap any single segment's contribution
    "ref_min_total_sec":     2.5,    # below this → borrow dominant speaker's ref
    "ref_min_quality":       6.0,    # score below this is logged as degraded
    "ref_pad_sec":           0.12,   # silence inserted between ref segments
    "ref_trim_db":           35,     # top_db for librosa.effects.trim on refs

    # ── XTTS-v2 synthesis

    # ── Cross-Lingual F5-TTS synthesis
    # The reference transcript is NOT used. infer_process_clf5 passes a dummy
    # string ("Useless here.") because the model was trained with the prompt
    # transcript removed — word boundaries come from forced alignment instead.
    # This is exactly the concern raised about feeding F5 a Devanagari
    # reference transcript: it does not arise here.
    #
    # Accent: CLF5 inherits F5's strong prompt-to-output accent transfer, which
    # is why v6.1 sounded Indian. Be honest in the paper — the checkpoint is
    # trained on Emilia (English + Chinese), NOT on Indian speech. Any Indian
    # accent comes from prompt transfer, not from Indian pretraining.
    "clf5_repo":             "QingyuLiu1/Cross-Lingual_F5-TTS",
    "clf5_ckpt":             "clf5_950000.safetensors",
    "clf5_rate_ckpt":        "syllables_gce_20000.safetensors",  # or phonemes_gce_14000
    "clf5_nfe_step":         32,     # flow-matching steps. 16 = ~2x faster,
                                     # slightly rougher. Drop to 16 if RTF hurts.
    "clf5_cfg_strength":     2.0,
    "clf5_sway":             -1.0,
    "clf5_cross_fade":       0.15,
    "clf5_seed":             1234,
    "clf5_fix_duration":     False,  # True = hand CLF5 an exact target length
                                     # instead of correcting afterwards. Serves
                                     # the fixed-length requirement directly,
                                     # but test on one clip before trusting it.
    "tts_chars_per_sec":     15.0,   # PLACEHOLDER — overwrite from CELL 10
    "tts_speed_lo":          0.85,
    "tts_speed_hi":          1.15,   # PLACEHOLDER — set from the CELL 10 sweep

    # ── CHANGE 3: NO-CLONING BASELINE ────────────────────────────────
    # When no_cloning is True every segment is synthesised from ONE fixed
    # generic English prompt instead of the segment speaker's own voice.
    # Nothing else changes: Demucs, ASR, diarization, MT, the time-lock,
    # silence-borrowing and the mix are all identical. That isolation is
    # the whole point — any metric difference is attributable to the
    # speaker prompt and to nothing else.
    #
    # IMPORTANT: speaker references are STILL extracted in this mode. They
    # are not used for synthesis, but SpeakerSim must still be measured
    # against the ORIGINAL SOURCE SPEAKER — that is the number that makes
    # the cloned run's 0.86 interpretable. Comparing the generic dub to the
    # generic prompt would measure nothing.
    "no_cloning":            False,  # flipped by the baseline runner cell
    "generic_voice_path":    None,   # set by CELL 3b
    "run_tag":               "",     # "" = cloned run; "_nocloning" = baseline
                                     # auto-derived if left empty

    # ── TRANSCRIPT FREEZE ────────────────────────────────────────────
    # Reuse the cloned run's ASR / MT / diarization / segment timings so the
    # two conditions are scored against byte-identical references. Without
    # this, Whisper and NLLB re-run and the baseline's WER_tts reference
    # text is no longer the same string the cloned run was scored against —
    # exactly the mismatched-component confound that forced the earlier
    # XTTS re-evaluation. Build the cache with CELL 8b before running.
    "freeze_segments":       True,
    "freeze_dir":            "frozen_segments",

    # Duration fitting (unchanged behaviour, renamed keys)
    "max_seg_sec":           8.0,
    "silence_borrow_frac":   0.85,
    "min_gap_keep_ms":       80,
    "never_drop_segments":   True,
    "comfort_speed":         1.20,
    "adaptive_fit":          True,
    "trim_lead_silence":     True,
    "overlap_max_s":         0.30,
    "segment_min_chars":     3,
    "fade_ms":               15,
    "gap_tolerance_s":       0.15,
    "junk_patterns": [
        "subscribe", "like and share", "like and subscribe",
        "share this video", "comment below", "press the bell",
        "सब्सक्राइब",
    ],
    "junk_max_chars":        60,

    # Mix
    "bg_duck_db":            -14,
    "target_lufs":           -24.0,
    "highpass_hz":           70,
    "out_sr":                44100,
}


In [ ]:
# ─────────────────────────────────────────
# CELL 3b — GENERIC ENGLISH VOICE  (CHANGE 3: no-cloning baseline)
# ─────────────────────────────────────────
# ONE fixed English prompt, used for every clip and every speaker in the
# baseline condition. Two properties matter for the paper:
#
#   1. It is FIXED. Same wav for hin_1 and tam_6, same wav for SPEAKER_00
#      and SPEAKER_03. "No speaker conditioning" means the prompt carries
#      no information about who is talking in the source.
#   2. Its LENGTH matches the cloned condition (ref_target_sec, 8 s).
#      CLF5 budgets ~22 s total across prompt + generated audio, so a
#      20 s prompt and an 8 s prompt do not leave the model the same room.
#      Matching the length keeps the comparison controlled rather than
#      confounding voice identity with prompt duration.
#
# Source order: explicit path -> f5-tts's bundled English sample -> upload.
# The bundled sample is the reproducible default: it ships with the pip
# package the paper already cites, so anyone can regenerate this exactly.
import os, glob, shutil
import numpy as np, librosa, soundfile as sf
from pathlib import Path

# Set this if you want to supply your own generic voice (a clean, neutral,
# single-speaker English wav). Leave None to auto-resolve.
GENERIC_VOICE_SRC = None

GEN_DIR = Path(OUTPUT_DIR) / "generic_voice"
GEN_DIR.mkdir(parents=True, exist_ok=True)
GEN_WAV = GEN_DIR / "generic_en.wav"

def _resolve_generic_source():
    if GENERIC_VOICE_SRC and os.path.exists(GENERIC_VOICE_SRC):
        return GENERIC_VOICE_SRC, "explicit path"
    # f5-tts ships an English reference sample with the package
    try:
        import f5_tts
        root = Path(f5_tts.__file__).parent
        for pat in ("**/basic_ref_en.wav", "**/*ref_en*.wav", "**/examples/**/*.wav"):
            hits = sorted(glob.glob(str(root / pat), recursive=True))
            if hits:
                return hits[0], "f5-tts bundled sample"
    except Exception as e:
        print(f"   (could not inspect f5_tts package: {e})")
    return None, None

_src, _origin = _resolve_generic_source()

if _src is None:
    print("No bundled English sample found. Upload one clean English wav "
          "(>= 8 s, one speaker, no music).")
    from google.colab import files
    up = files.upload()
    _src = list(up.keys())[0]
    _origin = "manual upload"

print(f"Generic voice source: {_src}\n   origin: {_origin}")

# ── Normalise to match the cloned condition's reference format ──
y, sr = librosa.load(_src, sr=None, mono=True)
y, _ = librosa.effects.trim(y, top_db=CFG["ref_trim_db"])
_target_n = int(CFG["ref_target_sec"] * sr)
if len(y) > _target_n:
    y = y[:_target_n]                      # cap, do not pad: same budget rule
peak = float(np.abs(y).max())
if peak > 0:
    y = y / peak * 0.95
sf.write(str(GEN_WAV), y.astype(np.float32), sr)

_q = ref_quality(y, sr)
CFG["generic_voice_path"] = str(GEN_WAV)

print(f"\n✅ Generic voice written: {GEN_WAV}")
print(f"   duration {len(y)/sr:.2f}s @ {sr} Hz")
print(f"   quality  score {_q['score']:.1f}  SNR {_q['snr_db']:.1f} dB  "
      f"flatness {_q['flatness']:.3f}")
if len(y) / sr < 3.0:
    print("   ⚠ shorter than 3 s — CLF5 clones poorly from very short prompts.")
if _q["score"] < CFG["ref_min_quality"]:
    print("   ⚠ quality below ref_min_quality. A noisy baseline prompt would "
          "understate the baseline unfairly — pick a cleaner clip.")
print("\n   LISTEN TO THIS before running the baseline. Report its provenance "
      "and duration in the paper — a reviewer will ask which voice it was.")


In [ ]:
# ─────────────────────────────────────────
# CELL 4 — SPEAKER REFERENCE EXTRACTION  (replaces CELL 4 + CELL 5 of v6.1)
# ─────────────────────────────────────────
# The whole voice bank and the gender/age detectors are gone. Each speaker's
# clone reference is cut from THEIR OWN Demucs vocals.
#
# Selection is quality-scored, not longest-first, because the reference is now
# separation output: residual music or reverb in the reference is copied into
# the cloned voice. ref_quality() is the guard — it is also written into the
# eval record so a bad MOS score can be traced back to a bad reference instead
# of being blamed on XTTS.
import numpy as np, librosa, soundfile as sf
from pathlib import Path


def ref_quality(y: np.ndarray, sr: int) -> dict:
    """
    Cheap, deterministic reference-audio quality score. Higher is better.
      snr_db   — speech frames vs floor frames; low ⇒ music/noise bleed
      flatness — spectral flatness on speech frames; high ⇒ noisy/musical
      clip     — fraction of clipped samples
    score = snr_db - 40*flatness - 200*clip
    """
    y = np.asarray(y, dtype=np.float32)
    if len(y) < int(0.4 * sr) or not np.isfinite(y).all():
        return {"score": -1e9, "snr_db": 0.0, "flatness": 1.0, "clip": 1.0}

    S = np.abs(librosa.stft(y, n_fft=1024, hop_length=256))
    rms = librosa.feature.rms(S=S, frame_length=1024)[0]
    if rms.max() <= 1e-8:
        return {"score": -1e9, "snr_db": 0.0, "flatness": 1.0, "clip": 1.0}

    thr = 0.35 * float(np.percentile(rms, 95))
    speech = rms > thr
    if speech.sum() < 5:
        return {"score": -1e9, "snr_db": 0.0, "flatness": 1.0, "clip": 0.0}
    floor = (float(np.median(rms[~speech])) if (~speech).sum() >= 5
             else float(np.percentile(rms, 10)))
    snr = 20.0 * np.log10((float(np.median(rms[speech])) + 1e-9) / (floor + 1e-9))

    flat = float(np.mean(librosa.feature.spectral_flatness(S=S, n_fft=1024)[0][speech]))
    clip = float(np.mean(np.abs(y) > 0.98))

    return {"score": float(snr - 40.0 * flat - 200.0 * clip),
            "snr_db": float(snr), "flatness": flat, "clip": clip}


def build_speaker_reference(spk: str, segs: list, vocals_np: np.ndarray,
                            sr: int, out_dir: Path) -> dict:
    """
    Cut the cleanest segments of `spk` out of the Demucs vocals stem, in
    descending quality order, until ref_target_sec is reached. Writes
    <out_dir>/<spk>.wav — LISTEN TO THESE before trusting any clone.
    """
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cands = []
    for s in segs:
        a = int(max(0.0, s.get("start", 0.0)) * sr)
        b = int(min(s.get("end", 0.0) * sr, len(vocals_np)))
        if b - a < int(CFG["ref_min_seg_sec"] * sr):
            continue
        y = vocals_np[a:int(min(b, a + CFG["ref_max_seg_sec"] * sr))]
        y, _ = librosa.effects.trim(y, top_db=CFG["ref_trim_db"])
        if len(y) < int(CFG["ref_min_seg_sec"] * sr):
            continue
        q = ref_quality(y, sr)
        cands.append((q["score"], s.get("start", 0.0), y, q))

    # Deterministic: quality first, then timestamp. Never random, never
    # dependent on how many times a cell has been run.
    cands.sort(key=lambda c: (-c[0], c[1]))

    pad = np.zeros(int(CFG["ref_pad_sec"] * sr), dtype=np.float32)
    picked, parts, total = [], [], 0.0
    for score, t0, y, q in cands:
        if total >= CFG["ref_target_sec"]:
            break
        parts.append(y.astype(np.float32)); parts.append(pad)
        total += len(y) / sr
        picked.append({"t": round(float(t0), 2), "sec": round(len(y) / sr, 2),
                       "score": round(score, 1), "snr_db": round(q["snr_db"], 1),
                       "flatness": round(q["flatness"], 3)})

    if not parts:
        return {"speaker": spk, "path": None, "total_sec": 0.0,
                "score": -1e9, "n_segments": 0, "picked": [], "degraded": True,
                "reason": "no usable segment"}

    ref = np.concatenate(parts)
    peak = float(np.abs(ref).max())
    if peak > 0:
        ref = ref / peak * 0.95
    path = out_dir / f"{spk}.wav"
    sf.write(str(path), ref, sr)

    agg = ref_quality(ref, sr)
    return {"speaker": spk, "path": str(path), "total_sec": round(total, 2),
            "score": round(agg["score"], 1), "snr_db": round(agg["snr_db"], 1),
            "flatness": round(agg["flatness"], 3),
            "n_segments": len(picked), "picked": picked[:8],
            "degraded": bool(total < CFG["ref_min_total_sec"]
                             or agg["score"] < CFG["ref_min_quality"]),
            "reason": ""}


print("✅ Reference extraction ready (no gender/age detection in this build)")


In [ ]:
# ─────────────────────────────────────────
# CELL 5 — LOAD HEAVY MODELS ONCE  (WhisperX + NLLB + Cross-Lingual F5-TTS)
# ─────────────────────────────────────────
import whisperx
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Loading WhisperX…")
whisper_model = whisperx.load_model(
    CFG["whisperx_model"], DEVICE,
    compute_type="float16" if DEVICE == "cuda" else "int8",
    vad_options={"vad_onset": 0.300, "vad_offset": 0.250},
)

print("Loading NLLB…")
nllb_tokenizer = AutoTokenizer.from_pretrained(CFG["nllb_model"])
nllb_model     = AutoModelForSeq2SeqLM.from_pretrained(
    CFG["nllb_model"],
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

# ── Cross-Lingual F5-TTS ──────────────────
# Architecture config must match the released checkpoint exactly; these values
# are the authors' own (from their Space's app.py), not guesses.
print("Loading Cross-Lingual F5-TTS…")
import sys
if CLF5_DIR not in sys.path:
    sys.path.insert(0, CLF5_DIR)
from cached_path import cached_path
from f5_tts.infer.utils_infer import load_model, load_vocoder, preprocess_ref_audio_text
from f5_tts.model import DiT
from utils_clf5_space import load_model_sp, infer_process_clf5

_repo = CFG["clf5_repo"]
clf5_vocoder = load_vocoder()
clf5_model = load_model(
    DiT,
    dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
    str(cached_path(f"hf://{_repo}/{CFG['clf5_ckpt']}")),
    vocab_file=str(cached_path(f"hf://{_repo}/vocab.txt")),
)
clf5_rate_model = load_model_sp(
    dict(dim=512, depth=6, heads=8, ff_mult=4),
    str(cached_path(f"hf://{_repo}/{CFG['clf5_rate_ckpt']}")),
    dict(target_sample_rate=24000, n_mel_channels=100, hop_length=256,
         win_length=1024, n_fft=1024, mel_spec_type="vocos"),
)

TTS_SR  = 24000
XTTS_SR = TTS_SR          # alias: CELL 8 still refers to XTTS_SR
print("✅ All models loaded")


def translate_batch(texts: list, src_code: str) -> list:
    """
    NOTE: v6.1 never set the tokenizer's src_lang, so NLLB silently treated
    every input as eng_Latn. Setting it is a genuine bug fix and will move the
    BLEU/chrF numbers — re-run the baseline before comparing against old ones.
    """
    nllb_tokenizer.src_lang = src_code
    inputs = nllb_tokenizer(texts, return_tensors="pt", padding=True,
                            truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        out_ids = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(CFG["tgt_lang"]),
            max_new_tokens=220, num_beams=2, length_penalty=0.9,
            no_repeat_ngram_size=CFG["no_repeat_ngram_size"],
            repetition_penalty=CFG["repetition_penalty"], early_stopping=True,
        )
    return nllb_tokenizer.batch_decode(out_ids, skip_special_tokens=True)


def collapse_repeats(text: str, max_run: int = None) -> str:
    max_run = max_run if max_run is not None else CFG["max_word_run"]
    out = []
    for w in text.split():
        if len(out) >= max_run and all(o.lower() == w.lower() for o in out[-max_run:]):
            continue
        out.append(w)
    return " ".join(out)


In [ ]:
# ─────────────────────────────────────────
# CELL 6 — DSP HELPERS  (unchanged from v6.1 — run before CELL 7)
# ─────────────────────────────────────────
import scipy.signal


def apply_fade(arr: np.ndarray, sr: int, fade_ms: int = 15) -> np.ndarray:
    """Raised-cosine fade in/out — removes the clicks at segment boundaries."""
    n = int(sr * fade_ms / 1000)
    if len(arr) < 2 * n or n < 2:
        return arr
    ramp = 0.5 * (1 - np.cos(np.linspace(0, np.pi, n, dtype=np.float32)))
    out = arr.copy()
    out[:n]  *= ramp
    out[-n:] *= ramp[::-1]
    return out


def highpass(arr: np.ndarray, sr: int, cutoff_hz: float = 70.0) -> np.ndarray:
    """Strip DC offset and TTS low-frequency rumble."""
    sos = scipy.signal.butter(2, cutoff_hz / (sr / 2), btype="highpass", output="sos")
    return scipy.signal.sosfilt(sos, arr).astype(np.float32)


def loudness_normalise(arr: np.ndarray, sr: int, target_lufs: float = -23.0) -> np.ndarray:
    """
    ONE loudness pass over the finished bus (replaces per-segment peak
    normalisation, which was inflating loudness range).
    """
    try:
        import pyloudnorm as pyln
        meter = pyln.Meter(sr)
        loud  = meter.integrated_loudness(arr)
        if np.isfinite(loud):
            arr = pyln.normalize.loudness(arr, loud, target_lufs)
    except Exception as e:
        print(f"   (pyloudnorm unavailable: {e} — falling back to peak norm)")
    peak = float(np.abs(arr).max())
    if peak > 0.99:
        arr = arr / peak * 0.99
    return arr.astype(np.float32)


def _match_length(arr: np.ndarray, target_len: int) -> np.ndarray:
    """Trim or zero-pad 1D array to target_len."""
    if len(arr) >= target_len:
        return arr[:target_len]
    return np.pad(arr, (0, target_len - len(arr)))


def cooperative_mix(speech: np.ndarray, bg: np.ndarray,
                    duck_db: float = -14,
                    frame_ms: int = 20, sr: int = 24000) -> np.ndarray:
    """
    Duck background by `duck_db` dB wherever speech energy is above threshold.
    Uses frame-level RMS with a short lookahead smoothing.
    """
    frame_len  = int(sr * frame_ms / 1000)
    duck_gain  = 10 ** (duck_db / 20)
    n_frames   = int(np.ceil(len(speech) / frame_len))
    gain_env   = np.ones(len(bg), dtype=np.float32)

    # Speech RMS threshold = 5% of peak RMS
    rms_vals = []
    for i in range(n_frames):
        s = speech[i*frame_len:(i+1)*frame_len]
        rms_vals.append(float(np.sqrt(np.mean(s**2) + 1e-9)))
    rms_thresh = np.percentile(rms_vals, 40) * 1.5  # adaptive

    for i, rms in enumerate(rms_vals):
        start = i * frame_len
        end   = min(start + frame_len, len(gain_env))
        if rms > rms_thresh:
            gain_env[start:end] = duck_gain

    # Smooth gain envelope (50 ms window) to avoid clicks
    smooth = int(sr * 0.05)
    gain_env = np.convolve(gain_env, np.ones(smooth)/smooth, mode='same')

    return bg * gain_env

In [ ]:
# ─────────────────────────────────────────
# CELL 7 — CROSS-LINGUAL F5-TTS SYNTHESIS HELPER
# ─────────────────────────────────────────
# No reference transcript anywhere. preprocess_ref_audio_text is given the
# dummy string the authors use; the model ignores it. No gender detection,
# no voice bank, no accent donors — the prompt is the speaker's own vocals.
import threading, os
import numpy as np, librosa, soundfile as sf, torchaudio

_tts_lock  = threading.Lock()      # CLF5 inference is not thread-safe
_REF_CACHE = {}                    # ref path → (preprocessed path, seconds)


def _prep_ref(ref_path: str):
    """f5-tts preprocessing (silence trim + clipping), cached per speaker."""
    if ref_path in _REF_CACHE:
        return _REF_CACHE[ref_path]
    with _tts_lock:
        prepped, _dummy = preprocess_ref_audio_text(
            ref_path, "Useless here.", show_info=lambda *a, **k: None)
    info = torchaudio.info(prepped)
    _REF_CACHE[ref_path] = (prepped, info.num_frames / info.sample_rate)
    return _REF_CACHE[ref_path]


def _looped(w: np.ndarray, sr: int) -> bool:
    """Autocorrelation check for the repeat-the-last-word failure."""
    if len(w) < sr:
        return False
    seg = w[-int(0.8 * sr):]
    seg = seg - seg.mean()
    ac  = np.correlate(seg, seg, "full")[len(seg) - 1:]
    ac  = ac / (ac[0] + 1e-9)
    return bool(ac[int(0.10 * sr):int(0.40 * sr)].max() > 0.6)


def synthesise_segment(text: str, ref: dict,
                       target_duration_s: Optional[float] = None):
    """Synthesise `text` in the voice of ref["path"]. Returns (float32, 24000).

    CHANGE 3: when CFG["no_cloning"] is set the speaker's own reference is
    replaced by the single generic English prompt. This is the ONLY place
    the baseline diverges from the cloned run — duration fitting, speed
    correction, the loop detector and the caller's time-lock logic are all
    untouched, so the two conditions differ in the prompt and nothing else.
    """
    if CFG.get("no_cloning"):
        _g = CFG.get("generic_voice_path")
        if not _g or not os.path.exists(_g):
            raise RuntimeError(
                "no_cloning is True but generic_voice_path is missing. "
                "Run CELL 3b first.")
        ref = {"path": _g, "borrowed_from": None, "generic": True}

    if len(text.strip()) < CFG["segment_min_chars"]:
        return np.zeros(int((target_duration_s or 0.5) * TTS_SR), np.float32), TTS_SR

    prepped, ref_sec = _prep_ref(ref["path"])

    def _run(speed, fix_dur=None):
        with _tts_lock:
            torch.manual_seed(CFG["clf5_seed"])   # flow matching samples noise
            wav, sr, _spec = infer_process_clf5(  # → pin it or runs differ
                clf5_rate_model, prepped, text, clf5_model, clf5_vocoder,
                show_info=lambda *a, **k: None, progress=None,
                nfe_step=CFG["clf5_nfe_step"],
                cfg_strength=CFG["clf5_cfg_strength"],
                sway_sampling_coef=CFG["clf5_sway"],
                cross_fade_duration=CFG["clf5_cross_fade"],
                speed=float(speed), fix_duration=fix_dur, device=DEVICE,
            )
        return np.asarray(wav, dtype=np.float32), sr

    speed, fix_dur = 1.0, None
    if target_duration_s is not None and target_duration_s > 0.2:
        if CFG.get("clf5_fix_duration", False):
            # fix_duration is TOTAL (prompt + generated); the prompt is sliced
            # off afterwards, so add ref_sec to get the target output length.
            fix_dur = ref_sec + target_duration_s
        else:
            ratio = (len(text) / CFG["tts_chars_per_sec"]) / target_duration_s
            speed = float(np.clip(ratio, CFG["tts_speed_lo"], CFG["tts_speed_hi"]))

    wav, sr = _run(speed, fix_dur)

    if speed > 1.05 and _looped(wav, sr):
        print(f"      ↻ loop detected @ speed {speed:.2f} — re-running at 1.0")
        wav, sr = _run(1.0, fix_dur)

    # Residual mismatch after the speed clamp: small phase-vocoder correction
    # only. Beyond ±25% is left alone — a short segment beats a metallic one.
    if target_duration_s is not None and target_duration_s > 0.2 and fix_dur is None:
        r = (len(wav) / sr) / target_duration_s
        if 1.08 < r < 1.45 or 0.72 < r < 0.92:
            wav = librosa.effects.time_stretch(wav, rate=r)

    if CFG.get("trim_lead_silence", True) and len(wav) > sr // 10:
        env, thr = np.abs(wav), 0.02 * np.abs(wav).max()
        first = int(np.argmax(env > thr))
        if 0 < first < len(wav) - sr // 20:
            wav = wav[first:]

    return apply_fade(wav, sr, CFG["fade_ms"]), sr


def synthesise_batch(segments_info: list) -> list:
    return [synthesise_segment(s["text"], s["ref"], s["target_duration_s"])
            for s in segments_info]


In [ ]:
# ─────────────────────────────────────────
# CELL 7b — QUICK ACCENT PROBE  (one sentence, ~15 s, no pipeline run)
# ─────────────────────────────────────────
# The single question this branch exists to answer: does CLF5 carry the Hindi
# speaker's accent into English? Answer it here before spending 6 minutes on a
# full dub. Run CELL 9 once first so references exist.
import glob, soundfile as sf
from pathlib import Path

REFS = sorted(p for p in glob.glob(f"{OUTPUT_DIR}/refs/*/*.wav") if "_blend" not in p)
assert REFS, "No references yet — run CELL 9 once, or point REF at any speaker wav."
print("\n".join(f"[{i}] {p}" for i, p in enumerate(REFS)))

TEXT = ("The state government announced a new policy for farmers, "
        "and officials confirmed the report on Tuesday morning.")
out_dir = Path(OUTPUT_DIR) / "accent_probe"; out_dir.mkdir(parents=True, exist_ok=True)

for i, ref in enumerate(REFS):
    w, sr = synthesise_segment(TEXT, {"path": ref}, target_duration_s=None)
    rms = float(np.sqrt((w ** 2).mean()))
    name = f"{i}_{Path(ref).stem}.wav"
    sf.write(str(out_dir / name), w, sr)
    print(f"   {name:28s} {len(w)/sr:5.2f}s  rms {rms:.4f}"
          f"{'   ⚠ NEAR-SILENT' if rms < 0.005 else ''}")
print(f"\nListen: {out_dir}  — compare against the v6.1 dub of the same speaker.")


In [ ]:
# ─────────────────────────────────────────
# CELL 8 — FULL PIPELINE FUNCTION  (v8, CLF5, any language → English)

# Whisper language code → NLLB-200 code. Whisper detects the language; this
# maps it to the translator. Add rows as needed — an unmapped language raises
# rather than silently mistranslating.
LANG_TO_NLLB = {
    # Indian
    "hi": "hin_Deva", "te": "tel_Telu", "ta": "tam_Taml", "kn": "kan_Knda",
    "ml": "mal_Mlym", "mr": "mar_Deva", "bn": "ben_Beng", "gu": "guj_Gujr",
    "pa": "pan_Guru", "or": "ory_Orya", "as": "asm_Beng", "ur": "urd_Arab",
    "ne": "npi_Deva", "si": "sin_Sinh", "sd": "snd_Arab",
    # Other
    "en": "eng_Latn", "es": "spa_Latn", "fr": "fra_Latn", "de": "deu_Latn",
    "pt": "por_Latn", "it": "ita_Latn", "ru": "rus_Cyrl", "ar": "arb_Arab",
    "fa": "pes_Arab", "tr": "tur_Latn", "zh": "zho_Hans", "ja": "jpn_Jpan",
    "ko": "kor_Hang", "vi": "vie_Latn", "th": "tha_Thai", "id": "ind_Latn",
}
# ─────────────────────────────────────────

def dub_video(input_video_path, output_path=None, hf_token=HF_TOKEN):
    import subprocess, re
    import pandas as pd
    t0 = time.time()
    video_path = Path(input_video_path)

    # CHANGE 3: every artefact this run writes is namespaced by _RUN_SUFFIX so
    # the baseline can never overwrite the cloned run's outputs. "" for the
    # cloned run keeps existing paths byte-identical to what you already have.
    _RUN_SUFFIX = CFG.get("run_tag") or ("_nocloning" if CFG.get("no_cloning") else "")
    if _RUN_SUFFIX:
        print(f"   [run tag: {_RUN_SUFFIX}]")

    # Baseline dubs go in their own folder so the 20 new mp4s are not
    # interleaved with the 20 cloned ones. The cloned run (_RUN_SUFFIX == "")
    # keeps writing to OUTPUT_DIR root exactly as before, so existing files and
    # the paths inside your current eval records stay valid.
    if _RUN_SUFFIX:
        _dub_dir = Path(OUTPUT_DIR) / f"dubbed{_RUN_SUFFIX}"
        _dub_dir.mkdir(parents=True, exist_ok=True)
    else:
        _dub_dir = Path(OUTPUT_DIR)

    if output_path is None:
        output_path = str(_dub_dir / f"{video_path.stem}_dubbed{_RUN_SUFFIX}.mp4")

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)

        # ── STEP 1: Extract audio
        print("\n[1/7] Extracting audio…")
        raw_audio = tmp / "raw.wav"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(video_path),
            "-vn", "-acodec", "pcm_s16le", "-ar", "44100", "-ac", "1",
            str(raw_audio)
        ], check=True, capture_output=True)

        # ── STEP 2: Demucs (run on CPU to avoid VRAM conflict)
        print("[2/7] Separating vocals (Demucs)…")
        import gc
        torch.cuda.empty_cache()
        gc.collect()
        demucs_cmd = [
            "python", "-m", "demucs",
            "--two-stems=vocals",
            "-n", CFG["demucs_model"],       # htdemucs_ft if refs sound muddy
            "-d", CFG["demucs_device"],
            "--segment", str(CFG["demucs_segment"]),
            "-o", str(tmp / "demucs"),
            str(raw_audio)
        ]
       # Demucs spawns child Python processes that inherit the environment.
        # An unset/empty PYTHONHASHSEED kills them before startup, so set it
        # explicitly here rather than depending on a cell having been run.
        _env = {**os.environ, "PYTHONHASHSEED": "0"}

        r = subprocess.run(demucs_cmd, capture_output=True, text=True, env=_env)
        if r.returncode != 0:
            print("   ⚠ Demucs failed on", CFG["demucs_device"], "— retrying on CPU")
            print("   stderr:", r.stderr[-1000:])
            demucs_cmd[demucs_cmd.index("-d") + 1] = "cpu"
            r2 = subprocess.run(demucs_cmd, capture_output=True, text=True, env=_env)
            if r2.returncode != 0:
                raise RuntimeError("Demucs failed on GPU and CPU:\n" + r2.stderr[-2000:])
        t_demucs = time.time()

        demucs_out  = tmp / "demucs" / CFG["demucs_model"] / "raw"
        vocals_path = demucs_out / "vocals.wav"
        bg_path     = demucs_out / "no_vocals.wav"

        vocals_np, vocals_sr = librosa.load(str(vocals_path), sr=None, mono=True)
        bg_np,     bg_sr     = librosa.load(str(bg_path),     sr=None, mono=True)
        # No raw-audio load any more: it existed only to profile F0 for gender
        # detection. Diarization still reads raw_audio from disk directly.

        import shutil
        shutil.copy(str(vocals_path),
                    f"{OUTPUT_DIR}/debug_vocals_{video_path.stem}{_RUN_SUFFIX}.wav")

        # ── STEP 3: WhisperX transcribe + align
        print("[3/7] Transcribing + diarizing (WhisperX)…")
        audio_for_whisper = whisperx.load_audio(str(vocals_path))
        _forced = CFG.get("asr_lang")
        result = whisper_model.transcribe(audio_for_whisper, batch_size=16,
                                          language=_forced)   # None → detect
        det_lang = (result.get("language") or _forced or "hi").lower()
        if det_lang not in LANG_TO_NLLB:
            raise RuntimeError(
                f"Detected language '{det_lang}' has no NLLB mapping. Add it to "
                f"LANG_TO_NLLB, or set CFG['asr_lang'] to force a known code.")
        src_code = LANG_TO_NLLB[det_lang]
        print(f"   language: {det_lang} → {src_code}"
              f"{' (forced)' if _forced else ' (auto-detected)'}")
        # ── Split over-long segments. WhisperX sometimes merges multiple
        # turns into one long segment (e.g. a 26s block spanning a
        # male→female→male change), keeping only one turn's text and
        # dropping the rest. Re-transcribe any segment longer than MAX_SEG
        # in fixed sub-windows and replace it.
        raw_audio_arr = whisperx.load_audio(str(raw_audio))
        def _safe_transcribe(clip, batch_size=8):
            """WhisperX on a speechless window returns zero segments, and newer
            transformers then indexes inputs[0] unconditionally → IndexError.
            Skip windows with no speech energy and swallow the empty case."""
            if len(clip) < 8000:
                return {"segments": []}
            if float(np.sqrt((np.asarray(clip, np.float32) ** 2).mean() + 1e-12)) < 0.002:
                return {"segments": []}
            try:
                return whisper_model.transcribe(clip, batch_size=batch_size,
                                                language=det_lang)
            except (IndexError, ValueError):
                return {"segments": []}

        MAX_SEG = CFG["max_seg_sec"]
        new_segments = []
        for s in result["segments"]:
            dur = s["end"] - s["start"]
            if dur <= MAX_SEG:
                new_segments.append(s); continue
            print(f"   ↳ splitting {dur:.1f}s segment at {s['start']:.1f}s")
            w = s["start"]
            while w < s["end"] - 1.0:
                w1 = min(w + MAX_SEG, s["end"])
                clip = raw_audio_arr[int(w*16000):int(w1*16000)]
                if len(clip) < 16000:
                    break
                rr = _safe_transcribe(clip)
                for sub in rr["segments"]:
                    new_segments.append({**sub,
                        "start": sub["start"] + w, "end": sub["end"] + w})
                w = w1
        new_segments.sort(key=lambda x: x["start"])
        result["segments"] = new_segments
        print(f"   ↳ segment count after splitting: {len(new_segments)}")

        # ── Gap-fill. The split pass only fixes segments that EXIST; spans
        # the main pass never transcribed stay silent. Find every span where
        # the clean vocals carry sustained speech energy but no segment
        # covers it, and force-transcribe it from raw. This is what "no
        # missing lines" actually requires.
        _v16 = librosa.resample(vocals_np, orig_sr=vocals_sr, target_sr=16000) \
               if vocals_sr != 16000 else vocals_np
        _hop = 1600                                    # 100 ms frames
        _rms = np.array([np.sqrt((_v16[j:j+_hop]**2).mean() + 1e-12)
                         for j in range(0, len(_v16), _hop)])
        _thr = max(float(np.percentile(_rms, 60)) * 0.5, 0.004)
        _speech = _rms > _thr
        _cov = np.zeros(len(_speech), bool)
        for s in result["segments"]:
            a = int(s["start"] * 10); b = int(s["end"] * 10) + 1
            _cov[a:min(b, len(_cov))] = True
        _gaps, _g0 = [], None
        for j in range(len(_speech)):
            if _speech[j] and not _cov[j]:
                if _g0 is None: _g0 = j
            else:
                if _g0 is not None and j - _g0 >= 15:   # ≥ 1.5 s of speech
                    _gaps.append((_g0/10.0, j/10.0))
                _g0 = None
        if _g0 is not None and len(_speech) - _g0 >= 15:
            _gaps.append((_g0/10.0, len(_speech)/10.0))
        for g0, g1 in _gaps:
            p0, p1 = max(0.0, g0 - 0.25), g1 + 0.25
            clip = raw_audio_arr[int(p0*16000):int(p1*16000)]
            if len(clip) < 8000: continue
            rr = _safe_transcribe(clip)
            for sub2 in rr["segments"]:
                result["segments"].append({**sub2, "start": sub2["start"]+p0,
                                           "end": sub2["end"]+p0})
            if rr["segments"]:
                print(f"   ↳ gap-fill {g0:.1f}-{g1:.1f}s: +{len(rr['segments'])} segs")
        result["segments"].sort(key=lambda x: x["start"])

        try:
            align_model, metadata = whisperx.load_align_model(
                language_code=det_lang, device=DEVICE)
        except Exception as _e:
            align_model = None
            print(f"   ⚠ no wav2vec2 alignment model for '{det_lang}' ({_e}) — "
                  f"continuing with segment-level timings only")
        if align_model is not None:
            result = whisperx.align(result["segments"], align_model, metadata,
                                    audio_for_whisper, DEVICE)
        if align_model is not None:
            del align_model
        gc.collect(); torch.cuda.empty_cache()

        # ── Diarization
        from pyannote.audio import Pipeline as PyannotePipeline
        diarize_pipeline = PyannotePipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1",
            token=hf_token
        ).to(torch.device(DEVICE))

        # NOTE: diarize the RAW audio, not the Demucs vocals stem.
        # Separation artifacts smear speaker embeddings and collapse speakers.
        diarize_out = diarize_pipeline(
            str(raw_audio),
            min_speakers=CFG["diar_min_speakers"],
            max_speakers=CFG["diar_max_speakers"],
        )

        # pyannote.audio 4.x returns a wrapper object, not an Annotation.
        # The Annotation lives at .speaker_diarization. Without this unwrap
        # every parser below silently returns [] and ALL segments collapse
        # onto one speaker — which is exactly what v5.1 was doing.
        if hasattr(diarize_out, "speaker_diarization"):
            diarize_out = diarize_out.speaker_diarization

        # Convert to whisperx-compatible dataframe — handles all pyannote output formats
        diarize_segments = []

        # Format 1: standard Annotation object
        if hasattr(diarize_out, 'itertracks'):
            for turn, _, speaker in diarize_out.itertracks(yield_label=True):
                diarize_segments.append({
                    "start": turn.start,
                    "end": turn.end,
                    "speaker": speaker
                })

        # Format 2: iterable of tuples
        if not diarize_segments:
            try:
                for item in diarize_out:
                    if isinstance(item, (list, tuple)) and len(item) == 3:
                        seg, _, spk = item
                        diarize_segments.append({
                            "start": seg.start,
                            "end": seg.end,
                            "speaker": spk
                        })
            except Exception:
                pass

        # Format 3: parse string representation
        if not diarize_segments:
            for line in str(diarize_out).split('\n'):
                m = re.match(r'\[\s*([\d.]+)\s*-->\s*([\d.]+)\]\s+\w+\s+(\w+)', line)
                if m:
                    diarize_segments.append({
                        "start": float(m.group(1)),
                        "end": float(m.group(2)),
                        "speaker": m.group(3)
                    })

        diarize_df = pd.DataFrame(diarize_segments) if diarize_segments else pd.DataFrame(columns=["start", "end", "speaker"])
        if diarize_df.empty:
            raise RuntimeError(
                "Diarization produced no segments. Every voice would collapse to one "
                "speaker, so stopping here rather than emitting a wrong dub.\n"
                f"  pipeline returned: {type(diarize_out).__name__}\n"
                "  If that type is not 'Annotation', pyannote changed its output "
                "format again — unwrap the new attribute above."
            )
        else:
            from collections import defaultdict as _dd
            _spk_dur = _dd(float)
            for _d in diarize_segments:
                _spk_dur[_d["speaker"]] += _d["end"] - _d["start"]
            print("   diarized speakers:",
                  {k: round(v, 1) for k, v in sorted(_spk_dur.items())})

        result = whisperx.assign_word_speakers(diarize_df, result)
        segments = result["segments"]
        print("   ── SEGMENT MAP ──")
        for _i, _s in enumerate(segments):
            print(f"     seg {_i}: {_s['start']:6.1f}-{_s['end']:6.1f}s  "
                  f"spk={_s.get('speaker','?'):11s}  {_s.get('text','').strip()[:50]!r}")

        # ── Fix silent collapse: whisperx sometimes fails to label segments
        # (music-heavy clips, turns not overlapping ASR segments). Previously
        # every unlabelled segment fell back to the DOMINANT speaker, which is
        # how an entire video ended up in one voice. Assign by maximum
        # temporal overlap with the diarization turns instead.
        def _best_overlap_speaker(s0, s1):
            best, best_ov = None, 0.0
            for d in diarize_segments:
                ov = min(s1, d["end"]) - max(s0, d["start"])
                if ov > best_ov:
                    best, best_ov = d["speaker"], ov
            if best is None:      # no overlap at all — take the nearest turn
                nd = min(diarize_segments,
                         key=lambda d: min(abs(d["start"] - s1), abs(s0 - d["end"])))
                best = nd["speaker"]
            return best

        _relabelled = 0
        for s in segments:
            if not s.get("speaker"):
                s["speaker"] = _best_overlap_speaker(s.get("start", 0.0),
                                                     s.get("end", 0.0))
                _relabelled += 1
        if _relabelled:
            print(f"   ↳ {_relabelled}/{len(segments)} segments had no label — "
                  f"assigned by temporal overlap with diarization turns")
        print(f"   → {len(segments)} segments, "
              f"{len(set(s.get('speaker','?') for s in segments))} speakers")
        t_asr = time.time()

        # ── STEP 4: Translate
        print("[4/7] Translating (NLLB-200)…")
        texts_src = [s["text"].strip() for s in segments]
        texts_en = []
        BATCH = 16
        for i in range(0, len(texts_src), BATCH):
            texts_en.extend(translate_batch(texts_src[i:i+BATCH], src_code))

        # Post-hoc repetition guard. A segment that trips this had garbage
        # source-language ASR — worth surfacing, not silently cleaning.
        cleaned = []
        for i, en in enumerate(texts_en):
            fixed = collapse_repeats(en)
            if fixed != en:
                print(f"   ⚠ REPEAT LOOP seg {i}: {en[:70]!r}")
                print(f"     → collapsed to: {fixed[:70]!r}")
                print(f"     source        : {texts_src[i][:70]!r}")
            cleaned.append(fixed)
        texts_en = cleaned

        for seg, en in zip(segments, texts_en):
            seg["text_en"] = en

        # ── TRANSCRIPT FREEZE  (CHANGE 3) ─────────────────────────
        # Load the cloned run's ASR / MT / diarization / timings verbatim if a
        # cache exists, so the baseline is not scored against a re-decoded
        # reference. If none exists, write one — the first run becomes the
        # frozen reference for every later condition.
        #
        # What this pins: segment boundaries, speaker labels, source text,
        # English MT text, detected language. What it does NOT pin: the
        # Demucs stems (re-separated each run). Separation is deterministic
        # for a fixed model and input, but note it in the paper's
        # reproducibility paragraph rather than claiming bit-exactness.
        if CFG.get("freeze_segments"):
            _frz_dir = Path(OUTPUT_DIR) / CFG.get("freeze_dir", "frozen_segments")
            _frz_dir.mkdir(parents=True, exist_ok=True)
            _frz = _frz_dir / f"{video_path.stem}.json"

            if _frz.exists():
                _f = json.load(open(_frz, encoding="utf-8"))
                _fsegs = _f.get("segments", [])
                if not _fsegs:
                    raise RuntimeError(f"Frozen cache {_frz} has no segments.")
                if len(_fsegs) != len(segments):
                    print(f"   ⚠ FROZEN CACHE OVERRIDES THIS RUN: "
                          f"{len(segments)} fresh segments -> {len(_fsegs)} frozen. "
                          f"This is expected and is what makes the two conditions "
                          f"comparable.")
                segments = [{
                    "start":   float(s["start"]),
                    "end":     float(s["end"]),
                    "speaker": s.get("speaker") or "SPEAKER_00",
                    "text":    s.get("text", ""),
                    "text_en": s.get("text_en", ""),
                } for s in _fsegs]
                texts_src = [s["text"]    for s in segments]
                texts_en  = [s["text_en"] for s in segments]
                det_lang  = _f.get("det_lang", det_lang)
                src_code  = _f.get("src_code", src_code)
                print(f"   ❄ frozen transcripts loaded: {_frz.name} "
                      f"({len(segments)} segments, lang {det_lang})")
            else:
                json.dump({
                    "clip_id":  video_path.stem,
                    "det_lang": det_lang,
                    "src_code": src_code,
                    "segments": [{
                        "start":   round(float(s.get("start", 0.0)), 3),
                        "end":     round(float(s.get("end", 0.0)), 3),
                        "speaker": s.get("speaker", "SPEAKER_00"),
                        "text":    s.get("text", ""),
                        "text_en": s.get("text_en", ""),
                    } for s in segments],
                }, open(_frz, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
                print(f"   ❄ froze transcripts -> {_frz.name} "
                      f"({len(segments)} segments)")

        # ── STEP 5: Build one clone reference per speaker, from that
        # speaker's OWN separated vocals. No gender detection, no age
        # detection, no voice bank — so gender misclassification is
        # structurally impossible rather than merely rarer.
        #
        # Note this also removes the need for the v6.1 speaker-merge hack:
        # if diarization over-splits one person into two clusters, both
        # clusters now clone the same person and sound identical.
        print("[5/7] Building speaker references (source cloning)…")
        from collections import defaultdict
        spk_segs  = defaultdict(list)
        spk_total = defaultdict(float)
        for seg in segments:
            spk = seg.get("speaker") or "SPEAKER_00"
            spk_segs[spk].append(seg)
            spk_total[spk] += max(0.0, seg.get("end", 0.0) - seg.get("start", 0.0))

        ref_dir = Path(OUTPUT_DIR) / f"refs{_RUN_SUFFIX}" / video_path.stem
        ref_stats, speaker_refs = {}, {}
        for spk in sorted(spk_segs, key=lambda k: -spk_total[k]):
            ref_stats[spk] = build_speaker_reference(
                spk, spk_segs[spk], vocals_np, vocals_sr, ref_dir)

        # Donor for speakers with too little (or unusable) clean audio: the
        # best-scoring reference in the clip. Logged, never silent.
        _usable = {k: v for k, v in ref_stats.items()
                   if v["path"] and v["total_sec"] >= CFG["ref_min_total_sec"]}
        _donor = max(_usable, key=lambda k: _usable[k]["score"]) if _usable else None

        for spk, info in ref_stats.items():
            if info["path"] and info["total_sec"] >= CFG["ref_min_total_sec"]:
                speaker_refs[spk] = {"path": info["path"], "borrowed_from": None}
            elif _donor:
                speaker_refs[spk] = {"path": ref_stats[_donor]["path"],
                                     "borrowed_from": _donor}
                info["reason"] = f"only {info['total_sec']:.1f}s clean audio"
                info["degraded"] = True
                print(f"   ⚠ {spk}: {info['reason']} → borrowing {_donor}'s reference")
            else:
                raise RuntimeError(
                    "No speaker has enough clean vocal audio to clone from. "
                    "Check outputs/debug_vocals_*.wav — separation probably failed."
                )

        for spk, info in ref_stats.items():
            flag = " ⚠ DEGRADED" if info["degraded"] else ""
            print(f"   {spk}: {info['total_sec']:5.1f}s ref from "
                  f"{info['n_segments']:2d} segs  score {info['score']:6.1f}  "
                  f"SNR {info.get('snr_db', 0):5.1f} dB  "
                  f"({spk_total[spk]:.1f}s spoken){flag}")
        print(f"   → references written to {ref_dir} — LISTEN TO THESE if MOS "
              f"comes back low.")
        t_profile = time.time()

        # ── STEP 6: Synthesise
        if CFG.get("no_cloning"):
            print("[6/7] Synthesising dubbed audio (CLF5, NO-CLONING BASELINE)…")
            print(f"   generic prompt: {CFG.get('generic_voice_path')}")
            print("   speaker references above were still built — they are the "
                  "SpeakerSim targets, not synthesis prompts.")
        else:
            print("[6/7] Synthesising dubbed audio (Cross-Lingual F5-TTS)…")
        # WhisperX is done — free its VRAM before CLF5 runs.
        gc.collect(); torch.cuda.empty_cache()
        total_dur = len(vocals_np) / vocals_sr
        dub_np    = np.zeros(int(total_dur * XTTS_SR), dtype=np.float32)

        dropped  = []
        filtered = []
        _fallback_spk = max(spk_total, key=spk_total.get) if spk_total else None

        # Precompute each segment's available slot = its own span PLUS the
        # silence gap to the next segment. Borrowing that gap is the only
        # slack we have while keeping the video the SAME LENGTH: speech can
        # expand into the pause after it without pushing the timeline.
        _starts = [seg.get("start", 0.0) for seg in segments]
        _ends   = [seg.get("end", s + 1.0) for seg, s in zip(segments, _starts)]
        _slot   = []
        for i in range(len(segments)):
            base = _ends[i] - _starts[i]
            gap  = (_starts[i+1] - _ends[i]) if i+1 < len(segments) else 0.4
            gap  = max(0.0, gap)
            borrow = min(gap * CFG["silence_borrow_frac"],
                         gap - CFG["min_gap_keep_ms"]/1000.0) if gap > CFG["min_gap_keep_ms"]/1000.0 else 0.0
            _slot.append(base + max(0.0, borrow))

        for i, seg in enumerate(segments):
            spk = seg.get("speaker") or "SPEAKER_00"
            if spk not in speaker_refs:
                spk = _fallback_spk
            ref     = speaker_refs[spk]
            text_en = seg.get("text_en", "").strip()
            hi_raw  = seg.get("text", "").strip()
            start_s = _starts[i]
            base_s  = _ends[i] - _starts[i]     # segment's OWN span (no borrow)
            slot_s  = _slot[i]                  # span + borrowable silence

            # ── Junk filter — SHORT standalone promo lines only. A word like
            # "channel" inside a real sentence must never kill the sentence;
            # v5.8 did exactly that and silently lost news content.
            _low = (text_en + " " + hi_raw).lower()
            if (len(text_en) < CFG["junk_max_chars"]
                    and any(p in _low for p in CFG.get("junk_patterns", []))):
                print(f"   ✖ seg {i} filtered (standalone promo): {text_en[:40]!r}")
                filtered.append((i, start_s, _ends[i], "junk"))
                continue

            # ── Adaptive fit. Decide per-segment whether this one needs help.
            #   Fits comfortably (natural pace ok) → use its OWN span, no borrow,
            #     no over-speed. Restores news-clip quality.
            #   Overflows → borrow silence + allow higher speed, and only trim
            #     if STILL overlong. Keeps monologue coverage.
            fit_dur = base_s
            if text_en and CFG.get("adaptive_fit", True):
                est = len(text_en) / CFG["tts_chars_per_sec"]
                speed_needed = est / max(base_s, 1e-3)
                if speed_needed <= CFG["comfort_speed"]:
                    fit_dur = base_s                       # comfortable — leave it alone
                else:
                    fit_dur = slot_s                       # overflow — grant borrowed silence
                    max_fit = slot_s * CFG["tts_speed_hi"]
                    if est > max_fit * 1.15:               # still too long → trim tail
                        keep = int(len(text_en) * (max_fit / est))
                        cut = text_en[:keep].rsplit(" ", 1)[0]
                        if len(cut) >= CFG["segment_min_chars"]:
                            print(f"   ✂ seg {i} trimmed ({len(text_en)}→{len(cut)} chars)")
                            text_en = cut
            else:
                fit_dur = slot_s

            if not text_en:
                # never silently drop — retry once on RAW hindi via a direct
                # re-translation, else place a short marker of silence so the
                # timeline stays intact and the gap is logged, not hidden.
                hi = seg.get("text", "").strip()
                if hi and CFG.get("never_drop_segments", True):
                    try:
                        text_en = translate_batch([hi])[0].strip()
                    except Exception:
                        text_en = ""
                if not text_en:
                    print(f"   ⚠ EMPTY seg {i} [{start_s:.2f}s] hi={seg.get('text','')[:30]!r}")
                    dropped.append((i, start_s, _ends[i], "empty_after_retry"))
                    continue

            try:
                wav, sr = synthesise_segment(text_en, ref, target_duration_s=fit_dur)
            except Exception as e:
                print(f"   ⚠ TTS FAILED seg {i}: {type(e).__name__}: {e}")
                dropped.append((i, start_s, _ends[i], f"tts_error:{type(e).__name__}"))
                continue
            if sr != XTTS_SR:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=XTTS_SR)
            pos = int(start_s * XTTS_SR)
            # Never collide with the next line: at most overlap_max_s into
            # the next segment's start, hard-faded past that. This is what
            # produced two voices talking over each other in v5.8.
            if i + 1 < len(segments):
                _lim = int((_starts[i+1] + CFG["overlap_max_s"]) * XTTS_SR)
                if pos + len(wav) > _lim and _lim - pos > 2400:
                    wav = apply_fade(wav[: _lim - pos], XTTS_SR, CFG["fade_ms"])
            end_p = pos + len(wav)
            if end_p > len(dub_np):
                dub_np = np.pad(dub_np, (0, end_p - len(dub_np)))
            dub_np[pos:end_p] += wav
            if (i + 1) % 10 == 0:
                print(f"   {i+1}/{len(segments)} segs | {time.time()-t0:.0f}s elapsed")

        # ── Coverage report: how much original speech has no dub over it?
        if dropped:
            lost = sum(e - s for _, s, e, _ in dropped)
            print(f"\n   ⚠ {len(dropped)} segments dropped, {lost:.1f}s of speech "
                  f"has NO dub ({lost/total_dur*100:.1f}% of the clip):")
            for i_, s_, e_, why in dropped:
                print(f"       seg {i_:>3}  {s_:7.2f}–{e_:7.2f}s  {why}")
        else:
            print("   ✅ all segments synthesised — no silent holes")

        # High-pass only here; loudness is handled once, on the final mix.
        dub_np = highpass(dub_np, XTTS_SR, CFG["highpass_hz"])
        t_tts  = time.time()

        # ── STEP 7: Mix + remux
        print("[7/7] Mixing + remuxing…")
        dub_wav = tmp / "dub_track.wav"
        mix_wav = tmp / "mix_final.wav"
        sf.write(str(dub_wav), dub_np, XTTS_SR)

        bg_24k   = librosa.resample(bg_np, orig_sr=bg_sr, target_sr=XTTS_SR)
        bg_24k   = _match_length(bg_24k, len(dub_np))
        bg_mixed = cooperative_mix(dub_np, bg_24k, duck_db=CFG["bg_duck_db"], sr=XTTS_SR)
        final_mix = dub_np + bg_mixed

        # Upsample the finished mix to the source rate, then ONE loudness pass.
        out_sr = CFG["out_sr"]
        if out_sr != XTTS_SR:
            final_mix = librosa.resample(final_mix, orig_sr=XTTS_SR, target_sr=out_sr,
                                         res_type="soxr_hq")
        final_mix = loudness_normalise(final_mix, out_sr, CFG["target_lufs"])
        final_mix = np.stack([final_mix, final_mix], axis=-1)   # mono → stereo
        sf.write(str(mix_wav), final_mix, out_sr)

        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(video_path),
            "-i", str(mix_wav),
            "-c:v", "copy",
            "-c:a", "aac", "-b:a", "192k",
            "-map", "0:v:0", "-map", "1:a:0",
            output_path
        ], check=True, capture_output=True)

    total_time = time.time() - t0
    rtf = total_time / total_dur
    stage_times = {
        "demucs":    t_demucs  - t0,
        "asr+diar":  t_asr     - t_demucs,
        "profile":   t_profile - t_asr,
        "tts":       t_tts     - t_profile,
        "mix+mux":   time.time() - t_tts,
    }
    print("\n   Per-stage RTF:")
    for k, v in stage_times.items():
        print(f"     {k:10s} {v:7.1f}s   {v/total_dur:5.2f}x")
    print(f"\n✅ Done! Output: {output_path}")
    print(f"   Video dur : {total_dur:.1f}s")
    print(f"   Wall time : {total_time:.1f}s")
    print(f"   RTF       : {rtf:.2f}x")

    # ── EVAL RECORD (new) ──────────────────────────────────
    import json as _json
    eval_dir = Path(OUTPUT_DIR) / f"eval_records{_RUN_SUFFIX}"
    eval_dir.mkdir(exist_ok=True)

    # ── PERSIST SPEAKER REFERENCE WAVS ─────────────────────────
    # The reference clips live in a TemporaryDirectory that is deleted when
    # this function returns, so "ref_path" in the record would dangle.
    # SpeakerSim needs them, so copy them somewhere permanent first.
    import shutil as _shutil
    ref_dir = Path(OUTPUT_DIR) / f"speaker_refs{_RUN_SUFFIX}"
    ref_dir.mkdir(exist_ok=True)
    _persisted = {}
    for _spk, _info in speaker_refs.items():
        _p = _info.get("path")
        if _p and os.path.exists(_p):
            _dst = ref_dir / f"{video_path.stem}__{_spk}.wav"
            try:
                _shutil.copy(_p, _dst)
                _persisted[_spk] = str(_dst)
            except Exception as _e:
                print(f"   ! could not persist ref for {_spk}: {_e}")
                _persisted[_spk] = None
        else:
            _persisted[_spk] = None

    eval_record = {
        "clip_id": video_path.stem,
        "hindi_ref": "",        # <-- fill by hand later (ground truth source lang)
        "english_ref": "",      # <-- fill by hand later (ground truth English)
        "source_asr_text": " ".join(texts_src),
        "english_mt_text": " ".join(texts_en),
        "final_dubbed_audio_path": output_path,
        "processing_time_sec": total_time,
        "audio_duration_sec": total_dur,
        "rtf": rtf,
        "stage_times": stage_times,
        "dropped_segments": dropped,
        "filtered_segments": filtered,
        "coverage_pct": round(100.0 * (1.0 -
                        (sum((d[2]-d[1]) for d in dropped) +
                         sum((f[2]-f[1]) for f in filtered)) /
                        max(total_dur, 1e-6)), 1),
        "tts": ("cross_lingual_f5_generic_voice" if CFG.get("no_cloning")
                else "cross_lingual_f5_source_clone"),

        # ── CHANGE 3 provenance ────────────────────────────────────
        # condition:      which arm of the comparison this record belongs to
        # synthesis_ref:  the prompt actually fed to CLF5
        # speaker_refs:   UNCHANGED meaning in both arms — the source
        #                 speaker's own audio, i.e. the SpeakerSim target
        "condition":         "no_cloning" if CFG.get("no_cloning") else "cloned",
        "run_tag":           _RUN_SUFFIX,
        "synthesis_ref":     (CFG.get("generic_voice_path")
                              if CFG.get("no_cloning") else "per_speaker_source"),
        "frozen_segments":   bool(CFG.get("freeze_segments")),
        "src_language": det_lang,
        "nllb_src_code": src_code,
        "nfe_step": CFG["clf5_nfe_step"],
        "demucs_model": CFG["demucs_model"],
        "n_segments": len(segments),

        # ── PER-SEGMENT RECORD (new in BATCH version) ──────────────
        # Needed for: time-lock / duration error, segment-level BLEU,
        # and locating which segments translated or synthesised badly.
        # "text_en_ref" is the Stage-2 field to fill by hand.
        "segments": [
            {
                "idx": _i,
                "start": round(float(_s.get("start", 0.0)), 3),
                "end": round(float(_s.get("end", 0.0)), 3),
                "slot_sec": round(float(_s.get("end", 0.0)) - float(_s.get("start", 0.0)), 3),
                "speaker": _s.get("speaker", "?"),
                "text_src": _src,
                "text_en_mt": _en,
                "text_en_ref": "",
            }
            for _i, (_s, _src, _en) in enumerate(zip(segments, texts_src, texts_en))
        ],

        "speaker_refs": {
            k: {"ref_path": _persisted.get(k),
                "ref_path_tmp": speaker_refs[k]["path"],
                "borrowed_from": speaker_refs[k]["borrowed_from"],
                "ref_sec": v["total_sec"],
                "ref_score": v["score"],
                "ref_snr_db": v.get("snr_db"),
                "ref_flatness": v.get("flatness"),
                "n_ref_segments": v["n_segments"],
                "degraded": v["degraded"],
                "spoken_sec": round(spk_total[k], 1)}
            for k, v in ref_stats.items()
        },
    }

    record_path = eval_dir / f"{video_path.stem}_eval.json"
    with open(record_path, "w") as f:
        _json.dump(eval_record, f, indent=2, ensure_ascii=False)

    print(f"   Eval record saved: {record_path}")
    # ── END EVAL RECORD ─────────────────────────────────────

    return output_path

In [ ]:
# ─────────────────────────────────────────
# CELL 9 — INPUTS (videos zip + transcripts zip) then BATCH RUN
# ─────────────────────────────────────────
# Upload BOTH zips when prompted (you can select them together):
#     1. source videos   -> hin_1.mp4, tel_2.mp4, ...
#     2. transcripts     -> hin_1.txt,  tel_2.txt,  ...
# Files may sit at the zip root or inside folders; both are flattened.
import glob, os, zipfile, shutil, traceback, re
from pathlib import Path

INPUT_MODE = "zip"        # "zip" | "drive"

DRIVE_VIDEO_DIR = "/content/drive/MyDrive/Dubbing/Input Videos"
DRIVE_TRANS_DIR = "/content/drive/MyDrive/Dubbing/Transcripts"

VID_DIR   = "/content/batch_input/videos"
TRANS_DIR = "/content/batch_input/transcripts"
VIDEO_EXT = (".mp4", ".mkv", ".mov", ".webm", ".avi")
TEXT_EXT  = (".txt", ".srt", ".vtt", ".json", ".csv", ".tsv")

os.makedirs(VID_DIR, exist_ok=True)
os.makedirs(TRANS_DIR, exist_ok=True)

if INPUT_MODE == "zip":
    from google.colab import files
    print("Select BOTH zips (videos + transcripts)…")
    up = files.upload()

    stage = "/content/_unzip"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage, exist_ok=True)
    for z in up:
        try:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(stage)
            print(f"  extracted {z}")
        except zipfile.BadZipFile:
            print(f"  !! {z} is not a zip — skipped")

    # sort every extracted file into videos/ or transcripts/ by extension
    nv = nt = 0
    for p in glob.glob(f"{stage}/**/*", recursive=True):
        if "__MACOSX" in p or not os.path.isfile(p):
            continue
        b = os.path.basename(p)
        if b.startswith("."):
            continue
        low = b.lower()
        if low.endswith(VIDEO_EXT):
            shutil.copy(p, os.path.join(VID_DIR, b));   nv += 1
        elif low.endswith(TEXT_EXT):
            shutil.copy(p, os.path.join(TRANS_DIR, b)); nt += 1
    print(f"\nsorted: {nv} video file(s), {nt} transcript file(s)")

elif INPUT_MODE == "drive":
    for src, dst in [(DRIVE_VIDEO_DIR, VID_DIR), (DRIVE_TRANS_DIR, TRANS_DIR)]:
        if os.path.isdir(src):
            for p in glob.glob(f"{src}/*"):
                if os.path.isfile(p):
                    shutil.copy(p, dst)
        else:
            print(f"  ! not found: {src}")

vids = sorted(p for p in glob.glob(f"{VID_DIR}/*") if p.lower().endswith(VIDEO_EXT))
trs  = sorted(p for p in glob.glob(f"{TRANS_DIR}/*"))

print(f"\n{len(vids)} videos:")
for v in vids: print("   ", os.path.basename(v))
print(f"\n{len(trs)} transcripts:")
for t in trs:  print("   ", os.path.basename(t))

# match transcripts to clips by stem
vstems = {Path(v).stem for v in vids}
tstems = {Path(t).stem for t in trs}
if vstems - tstems:
    print(f"\n!! videos with NO transcript: {sorted(vstems - tstems)}")
if tstems - vstems:
    print(f"!! transcripts with NO video:  {sorted(tstems - vstems)}")

# keep transcripts on Drive too, so they survive a runtime restart
_keep = Path(OUTPUT_DIR) / "transcripts"
_keep.mkdir(parents=True, exist_ok=True)
for t in trs:
    shutil.copy(t, _keep / os.path.basename(t))
print(f"\ntranscripts copied to {_keep}")

if not vids:
    raise SystemExit("No videos found.")

# ── BATCH RUN (resumable; one failure does not kill the run) ───────
done, failed, skipped = [], [], []
for v in vids:
    cid     = Path(v).stem
    out_mp4 = f"{OUTPUT_DIR}/{cid}_dubbed.mp4"
    rec     = f"{OUTPUT_DIR}/eval_records/{cid}_eval.json"
    if os.path.exists(out_mp4) and os.path.exists(rec):
        print(f"[skip] {cid}"); skipped.append(cid); continue

    print(f"\n{'='*64}\n>>  {cid}\n{'='*64}")
    try:
        dub_video(v, hf_token=HF_TOKEN)
        done.append(cid)
    except Exception as e:
        failed.append((cid, repr(e)[:300]))
        print(f"\n!! FAILED {cid}: {e}")
        traceback.print_exc()
    finally:
        torch.cuda.empty_cache(); gc.collect()

print(f"\n{'='*64}\ndone {len(done)} | skipped {len(skipped)} | failed {len(failed)}")
for c, e in failed: print(f"  {c}: {e}")


In [ ]:
# ─────────────────────────────────────────
# CELL 10 — CALIBRATE tts_chars_per_sec AND tts_speed_hi
# ─────────────────────────────────────────
# CLF5 derives its own duration from a learned speaking-rate predictor, so its
# pace tracks the PROMPT speaker rather than being fixed. Calibrate on a real
# reference, not a studio clip, and re-run if you change ref_target_sec.
import glob, numpy as np, soundfile as sf
from pathlib import Path

REF_WAV = None
if REF_WAV is None:
    _c = sorted(glob.glob(f"{OUTPUT_DIR}/refs/*/*.wav"), key=os.path.getmtime)
    assert _c, "No reference wavs yet — run CELL 9 first."
    REF_WAV = _c[-1]
print(f"Reference: {REF_WAV}")

CALIB_TEXT = [
    "The state government announced a new policy for farmers this morning.",
    "Police have registered a case and started an investigation into the matter.",
    "Heavy rainfall is expected across the region over the next two days.",
    "The minister said the project would be completed before the end of the year.",
    "Officials confirmed that all passengers on board were reported safe.",
]

def _raw(text, speed):
    w, sr = synthesise_segment(text, {"path": REF_WAV}, target_duration_s=None) \
            if speed == 1.0 else (None, None)
    if w is None:
        _p, _rs = _prep_ref(REF_WAV)
        torch.manual_seed(CFG["clf5_seed"])
        w, sr, _ = infer_process_clf5(
            clf5_rate_model, _p, text, clf5_model, clf5_vocoder,
            show_info=lambda *a, **k: None, progress=None,
            nfe_step=CFG["clf5_nfe_step"], speed=float(speed), device=DEVICE)
        w = np.asarray(w, dtype=np.float32)
    w, _ = librosa.effects.trim(w, top_db=35)
    return w

print("\n[1/2] Measuring chars/sec at speed 1.0…")
chars = secs = 0
for t in CALIB_TEXT:
    w = _raw(t, 1.0); d = len(w) / TTS_SR
    chars += len(t); secs += d
    print(f"   {len(t):3d} chars → {d:5.2f}s  ({len(t)/d:5.2f} c/s)")
cps = chars / secs
print(f"   ⇒ tts_chars_per_sec = {cps:.2f}")

print("\n[2/2] Speed sweep — listen before trusting the numbers…")
sw = Path(OUTPUT_DIR) / "calib"; sw.mkdir(parents=True, exist_ok=True)
probe_t = CALIB_TEXT[3]
base = len(_raw(probe_t, 1.0)) / TTS_SR
clean = []
for sp in [1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30]:
    w = _raw(probe_t, sp); d = len(w) / TTS_SR
    loop = _looped(w, TTS_SR)
    sf.write(str(sw / f"speed_{sp:.2f}.wav"), w, TTS_SR)
    if not loop: clean.append(sp)
    print(f"   speed {sp:.2f} → {d:5.2f}s  effective {base/max(d,1e-6):4.2f}x"
          f"  {'⚠ LOOP/REPEAT' if loop else ''}")
rec = max(clean) if clean else 1.10
print(f"\nPaste into CELL 3:\n"
      f'    "tts_chars_per_sec":     {cps:.1f},\n'
      f'    "tts_speed_hi":          {rec:.2f},')
print(f"Listen to {sw}/speed_*.wav — the detector catches looping, not slurring.")


In [ ]:
# ─────────────────────────────────────────
# CELL 11 — COVERAGE / EVAL REPORT  (run any time after dubbing)
# ─────────────────────────────────────────
import json, glob, os
import pandas as pd

recs = sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json"))
print(f"{len(recs)} eval records\n")

rows = []
for p in recs:
    r = json.load(open(p))
    segs = r.get("segments", [])
    slots = [s["slot_sec"] for s in segs] if segs else []
    rows.append({
        "clip":      r["clip_id"],
        "lang":      r.get("src_language", "?"),
        "dur_s":     round(r.get("audio_duration_sec", 0)),
        "n_segs":    r.get("n_segments", len(segs)),
        "cover_%":   r.get("coverage_pct", None),
        "dropped":   len(r.get("dropped_segments", [])),
        "filtered":  len(r.get("filtered_segments", [])),
        "rtf":       round(r.get("rtf", 0), 2),
        "max_slot":  round(max(slots), 1) if slots else 0,
        "mean_slot": round(sum(slots)/len(slots), 1) if slots else 0,
        "degraded_refs": sum(1 for v in r.get("speaker_refs", {}).values()
                             if v.get("degraded")),
    })

df = pd.DataFrame(rows).sort_values("clip")
print(df.to_string(index=False))
df.to_csv(f"{OUTPUT_DIR}/eval_overview.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/eval_overview.csv")

print("\n--- FLAGS ---")
n = 0
for _, r in df.iterrows():
    f = []
    if r["cover_%"] is not None and r["cover_%"] < 80:
        f.append(f"low coverage {r['cover_%']}%")
    if r["dropped"] > 0:
        f.append(f"{r['dropped']} dropped segs")
    if r["degraded_refs"] > 0:
        f.append(f"{r['degraded_refs']} degraded speaker ref(s)")
    if r["max_slot"] > 15:
        f.append(f"long slot {r['max_slot']}s")
    if f:
        print(f"  {r['clip']}: " + "; ".join(f)); n += 1
if n == 0:
    print("  none")

# which clips are still missing outputs entirely
srcs = {os.path.splitext(os.path.basename(v))[0]
        for v in glob.glob("/content/drive/MyDrive/Dubbing/Input Videos/*.mp4")}
got = set(df["clip"]) if len(df) else set()
missing = sorted(srcs - got)
if missing:
    print(f"\nNO EVAL RECORD for: {missing}")


## Evaluation

Runs the nine metrics from `eval_metrics_summary.md`.

| Cell | Does |
|---|---|
| 12 | Loads human transcripts into the eval records (`source_ref`, `english_ref`) |
| 13 | Re-transcribes the dubbed audio (`dub_hyp`) |
| 14 | SpeakerSim from persisted reference clips |
| 15 | Computes all nine metrics + per-language aggregation |
| 16-17 | Export / re-import English references (only if BLEU is still blocked) |

**BLEU and chrF need a human English translation.** Source-language transcripts
alone enable WER_asr / CER_asr but not BLEU / chrF.


In [ ]:
# ─────────────────────────────────────────
# CELL 12 — EVAL 1/5: load human transcripts into the eval records
# ─────────────────────────────────────────
# Fills source_ref (and english_ref if the file contains an English section).
# Handles .txt / .srt / .vtt / .json / .csv, strips timecodes and speaker tags,
# and splits by script: Devanagari/Telugu/Tamil -> source_ref, Latin -> english_ref.
import glob, json, os, re, unicodedata

TRANS_SRC = f"{OUTPUT_DIR}/transcripts"

SCRIPT_RANGES = {
    "deva": (0x0900, 0x097F),
    "telu": (0x0C00, 0x0C7F),
    "taml": (0x0B80, 0x0BFF),
}

def _script_of(line):
    counts = {k: 0 for k in SCRIPT_RANGES}
    latin = 0
    for ch in line:
        o = ord(ch)
        for k, (a, b) in SCRIPT_RANGES.items():
            if a <= o <= b:
                counts[k] += 1
        if 'a' <= ch.lower() <= 'z':
            latin += 1
    indic = sum(counts.values())
    if indic == 0 and latin == 0:
        return None
    return "indic" if indic >= latin else "latin"

_TS = re.compile(r"^\s*(\d+\s*$|\d{1,2}:\d{2}(:\d{2})?([.,]\d+)?\s*(-->|-)?)")
_SPK = re.compile(r"^\s*(speaker\s*\d+|spk\s*\d+|[A-Z][a-z]+)\s*:\s*", re.I)

def _clean_lines(raw):
    out = []
    for ln in raw.splitlines():
        ln = ln.strip()
        if not ln or ln.upper() == "WEBVTT":
            continue
        if _TS.match(ln) or "-->" in ln:
            continue
        ln = _SPK.sub("", ln)
        if ln:
            out.append(ln)
    return out

def parse_transcript(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".json":
        d = json.load(open(path, encoding="utf-8"))
        if isinstance(d, dict):
            for k in ("text", "transcript", "transcription"):
                if isinstance(d.get(k), str):
                    return _clean_lines(d[k])
            for k in ("segments", "results", "words"):
                if isinstance(d.get(k), list):
                    return _clean_lines(" ".join(
                        str(s.get("text", "")) for s in d[k] if isinstance(s, dict)))
        elif isinstance(d, list):
            return _clean_lines(" ".join(
                str(s.get("text", s)) for s in d))
        return []
    if ext in (".csv", ".tsv"):
        import csv
        sep = "\t" if ext == ".tsv" else ","
        rows = list(csv.reader(open(path, encoding="utf-8", errors="replace"),
                              delimiter=sep))
        if not rows: return []
        hdr = [h.strip().lower() for h in rows[0]]
        ti = next((i for i, h in enumerate(hdr) if "text" in h or "transcript" in h), None)
        body = rows[1:] if ti is not None else rows
        ti = ti if ti is not None else (len(hdr) - 1)
        return _clean_lines(" ".join(r[ti] for r in body if len(r) > ti))
    raw = open(path, encoding="utf-8", errors="replace").read()
    return _clean_lines(raw)

def split_by_script(lines):
    src, eng = [], []
    for ln in lines:
        s = _script_of(ln)
        (eng if s == "latin" else src).append(ln)
    return " ".join(src).strip(), " ".join(eng).strip()

files = sorted(glob.glob(f"{TRANS_SRC}/*"))
print(f"{len(files)} transcript file(s) in {TRANS_SRC}\n")

updated, no_record, empty = [], [], []
for p in files:
    cid = os.path.splitext(os.path.basename(p))[0]
    rec = f"{OUTPUT_DIR}/eval_records/{cid}_eval.json"
    if not os.path.exists(rec):
        no_record.append(cid); continue

    lines = parse_transcript(p)
    src_txt, eng_txt = split_by_script(lines)
    if not src_txt and not eng_txt:
        empty.append(cid); continue

    r = json.load(open(rec, encoding="utf-8"))
    if src_txt:
        r["source_ref"] = src_txt
        r["hindi_ref"]  = src_txt          # legacy field name in the schema
    if eng_txt:
        r["english_ref"] = eng_txt
    r["transcript_file"] = os.path.basename(p)
    json.dump(r, open(rec, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    updated.append(cid)
    print(f"[ok] {cid:8s} source_ref {len(src_txt):6d} chars | "
          f"english_ref {len(eng_txt):6d} chars")

print(f"\nupdated {len(updated)}")
if no_record: print(f"no eval record for: {no_record}")
if empty:     print(f"parsed empty: {empty}")

n_eng = sum(1 for c in updated
            if json.load(open(f"{OUTPUT_DIR}/eval_records/{c}_eval.json",
                              encoding="utf-8")).get("english_ref", "").strip())
print(f"\nclips with english_ref: {n_eng}/{len(updated)}")
if n_eng == 0:
    print("  -> BLEU/chrF cannot be computed. They need a HUMAN English")
    print("     translation. Source-language transcripts alone enable")
    print("     WER_asr/CER_asr but not BLEU/chrF.")


In [ ]:
# ─────────────────────────────────────────
# CELL 13 — EVAL 2/5: re-transcribe the dubbed audio  (dub_hyp)
# ─────────────────────────────────────────
# Needed for WER_tts / CER_tts. Uses the WhisperX already loaded in CELL 5.
!pip install -q jiwer sacrebleu resemblyzer

import glob, json, os, subprocess, tempfile

recs = sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json"))
print(f"{len(recs)} eval records\n")

for p in recs:
    r = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]
    if r.get("dub_hyp"):
        print(f"[skip] {cid}"); continue

    dub = r.get("final_dubbed_audio_path") or f"{OUTPUT_DIR}/{cid}_dubbed.mp4"
    if not os.path.exists(dub):
        print(f"[miss] {cid}: {dub}"); continue

    with tempfile.TemporaryDirectory() as td:
        w = os.path.join(td, "d.wav")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", dub,
                        "-vn", "-ac", "1", "-ar", "16000", w], check=True)
        res = whisper_model.transcribe(whisperx.load_audio(w),
                                       batch_size=8, language="en")

    segs = res.get("segments", [])
    r["dub_hyp"] = " ".join(s["text"].strip() for s in segs)
    r["dub_hyp_segments"] = [{"start": round(float(s.get("start", 0)), 3),
                              "end":   round(float(s.get("end", 0)), 3),
                              "text":  s["text"].strip()} for s in segs]
    json.dump(r, open(p, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"[ok]   {cid}: {len(segs)} segments")

print("\ndub_hyp written into every record.")


In [ ]:
# ─────────────────────────────────────────
# CELL 14 — EVAL 3/5: SpeakerSim  (metric #4)
# ─────────────────────────────────────────
# Cosine similarity between the Resemblyzer embedding of the source speaker's
# persisted reference clip and that same speaker's segments in the dub.
# Requires the CELL 8 patch that copies reference wavs out of the temp dir,
# so it only works for clips dubbed with this notebook.
import glob, json, os, subprocess, tempfile, warnings
warnings.filterwarnings("ignore")
import numpy as np, soundfile as sf
from resemblyzer import VoiceEncoder, preprocess_wav

_enc = VoiceEncoder(verbose=False)

def _embed_file(p):
    try:
        return _enc.embed_utterance(preprocess_wav(p))
    except Exception:
        return None

def _embed_array(y, sr=16000):
    try:
        if np.sqrt(np.mean(y ** 2)) < 1e-4:
            return None
        return _enc.embed_utterance(preprocess_wav(y, source_sr=sr))
    except Exception:
        return None

def _cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

for p in sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json")):
    r   = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]
    if r.get("speaker_sim") is not None:
        print(f"[skip] {cid}"); continue

    refs = r.get("speaker_refs", {})
    dub  = r.get("final_dubbed_audio_path") or f"{OUTPUT_DIR}/{cid}_dubbed.mp4"
    if not refs or not os.path.exists(dub):
        print(f"[miss] {cid}"); continue

    with tempfile.TemporaryDirectory() as td:
        w = os.path.join(td, "d.wav")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", dub,
                        "-vn", "-ac", "1", "-ar", "16000", w], check=True)
        y, sr = sf.read(w)
        if y.ndim > 1: y = y.mean(1)
        y = y.astype(np.float32)

        per_spk = {}
        for spk, info in refs.items():
            rp = info.get("ref_path")
            if not rp or not os.path.exists(rp):
                continue
            e_ref = _embed_file(rp)
            if e_ref is None:
                continue
            # concatenate this speaker's slices of the dub
            spans = [s for s in r.get("segments", []) if s.get("speaker") == spk]
            if spans:
                chunks = [y[int(s["start"] * sr):int(s["end"] * sr)] for s in spans]
                chunks = [c for c in chunks if len(c) > sr // 2]
                seg_audio = np.concatenate(chunks) if chunks else y
            else:
                seg_audio = y
            e_dub = _embed_array(seg_audio, sr)
            if e_dub is None:
                continue
            per_spk[spk] = round(_cos(e_ref, e_dub), 4)

    if per_spk:
        # duration-weighted where possible, else plain mean
        wts = {}
        for spk in per_spk:
            wts[spk] = sum(s["slot_sec"] for s in r.get("segments", [])
                           if s.get("speaker") == spk) or 1.0
        tot = sum(wts.values())
        r["speaker_sim"] = round(sum(per_spk[s] * wts[s] for s in per_spk) / tot, 4)
    else:
        r["speaker_sim"] = None
    r["speaker_sim_per_speaker"] = per_spk

    json.dump(r, open(p, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"[ok]   {cid}: SpeakerSim {r['speaker_sim']}  {per_spk}")


In [ ]:
# ─────────────────────────────────────────
# CELL 15 — EVAL 4/5: compute all nine metrics
# ─────────────────────────────────────────
#  1. WER_asr / CER_asr   pipeline_src  vs source_ref     (ASR accuracy)
#  2. BLEU / chrF         pipeline_eng  vs english_ref    (translation quality)
#  3. WER_tts / CER_tts   dub_hyp       vs pipeline_eng   (TTS intelligibility)
#  4. SpeakerSim          from CELL 14
#  5. Coverage            from the pipeline
#  6. RTF                 from the pipeline
import glob, json, re
import pandas as pd, jiwer, sacrebleu

def _n(t):
    t = str(t or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())

def _wer(ref, hyp):
    ref, hyp = _n(ref), _n(hyp)
    if not ref or not hyp: return None
    return round(jiwer.wer(ref, hyp), 4)

def _cer(ref, hyp):
    ref, hyp = _n(ref), _n(hyp)
    if not ref or not hyp: return None
    return round(jiwer.cer(ref, hyp), 4)

LANGNAME = {"hin": "Hindi", "tel": "Telugu", "tam": "Tamil"}

rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json")):
    r   = json.load(open(p, encoding="utf-8"))
    cid = r["clip_id"]

    src_ref = r.get("source_ref") or r.get("hindi_ref") or ""
    eng_ref = r.get("english_ref") or ""
    pipe_src = r.get("source_asr_text") or ""
    pipe_eng = r.get("english_mt_text") or ""
    dub_hyp  = r.get("dub_hyp") or ""

    bleu = chrf = None
    if pipe_eng.strip() and eng_ref.strip():
        bleu = round(sacrebleu.sentence_bleu(pipe_eng, [eng_ref]).score, 2)
        chrf = round(sacrebleu.sentence_chrf(pipe_eng, [eng_ref]).score, 2)

    rows.append({
        "clip":      cid,
        "lang":      LANGNAME.get(cid.split("_")[0], r.get("src_language", "?")),
        "WER_asr":   _wer(src_ref, pipe_src),
        "CER_asr":   _cer(src_ref, pipe_src),
        "BLEU":      bleu,
        "chrF":      chrf,
        "WER_tts":   _wer(pipe_eng, dub_hyp),
        "CER_tts":   _cer(pipe_eng, dub_hyp),
        "SpeakerSim": r.get("speaker_sim"),
        "Coverage_%": r.get("coverage_pct"),
        "RTF":       round(r.get("rtf", 0), 2) if r.get("rtf") else None,
    })

df = pd.DataFrame(rows).sort_values(["lang", "clip"]).reset_index(drop=True)
METRICS = ["WER_asr","CER_asr","BLEU","chrF","WER_tts","CER_tts",
           "SpeakerSim","Coverage_%","RTF"]

pd.set_option("display.width", 220)
print("PER-CLIP")
print("=" * 118)
print(df.to_string(index=False))

# ── Aggregation: Mean (all) then per language, per the metrics spec ──
agg = [df[METRICS].mean().round(3).to_dict() | {"clip": "Mean (all)", "lang": ""}]
for lg in ["Hindi", "Telugu", "Tamil"]:
    sub = df[df["lang"] == lg]
    if len(sub):
        agg.append(sub[METRICS].mean().round(3).to_dict()
                   | {"clip": f"Mean ({lg})", "lang": ""})

agg_df = pd.DataFrame(agg)[["clip", "lang"] + METRICS]
print("\nAGGREGATE")
print("=" * 118)
print(agg_df.to_string(index=False))

final = pd.concat([df, agg_df], ignore_index=True)
final.to_csv(f"{OUTPUT_DIR}/eval_metrics_CLF5.csv", index=False)
print(f"\nSaved: {OUTPUT_DIR}/eval_metrics_CLF5.csv")

# corpus BLEU — cite this, not the mean of per-clip BLEU
hyps, refs = [], []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json")):
    r = json.load(open(p, encoding="utf-8"))
    if (r.get("english_mt_text") or "").strip() and (r.get("english_ref") or "").strip():
        hyps.append(r["english_mt_text"]); refs.append(r["english_ref"])
if hyps:
    print(f"\nCORPUS BLEU {sacrebleu.corpus_bleu(hyps,[refs]).score:.2f}"
          f"  chrF {sacrebleu.corpus_chrf(hyps,[refs]).score:.2f}  (n={len(hyps)})")
else:
    print("\nCORPUS BLEU: not computable — no clip has a human english_ref.")

missing = {m: int(df[m].isna().sum()) for m in METRICS if df[m].isna().any()}
if missing:
    print(f"\nMissing values: {missing}")


In [ ]:
# ─────────────────────────────────────────
# CELL 16 — EVAL 5/5: export English references to fill in (for BLEU/chrF)
# ─────────────────────────────────────────
# Only needed if your transcripts were source-language only. BLEU and chrF
# cannot be computed without a HUMAN English translation — an MT output or a
# re-ASR of the dub will not do, since the same error then appears on both
# sides of the comparison and the score cannot detect it.
import glob, json
import pandas as pd

rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/eval_records/*_eval.json")):
    r = json.load(open(p, encoding="utf-8"))
    rows.append({
        "clip_id":     r["clip_id"],
        "lang":        r.get("src_language", "?"),
        "source_ref":  (r.get("source_ref") or r.get("hindi_ref") or "")[:32000],
        "english_ref": r.get("english_ref", ""),      # <-- FILL THIS
        "notes":       "",
    })
out = f"{OUTPUT_DIR}/english_refs_to_fill.csv"
pd.DataFrame(rows).to_csv(out, index=False)
print(f"{len(rows)} rows -> {out}")
print("Translate source_ref -> english_ref by hand, save as")
print(f"{OUTPUT_DIR}/english_refs_filled.csv, then run CELL 17.")


In [ ]:
# ─────────────────────────────────────────
# CELL 17 — load filled English references back in
# ─────────────────────────────────────────
import pandas as pd, json, os

FILLED = f"{OUTPUT_DIR}/english_refs_filled.csv"
if not os.path.exists(FILLED):
    print(f"Not found: {FILLED}")
else:
    df = pd.read_csv(FILLED).fillna("")
    n = 0
    for _, row in df.iterrows():
        ref = str(row.get("english_ref", "")).strip()
        if not ref: continue
        p = f"{OUTPUT_DIR}/eval_records/{row['clip_id']}_eval.json"
        if not os.path.exists(p):
            print(f"  no record: {row['clip_id']}"); continue
        r = json.load(open(p, encoding="utf-8"))
        r["english_ref"] = ref
        json.dump(r, open(p, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        n += 1
    print(f"Updated {n} record(s). Re-run CELL 15 to get BLEU/chrF.")


---

## CHANGE 3 — no-cloning baseline

One extra arm of the same pipeline: every segment synthesised from a single
fixed generic English prompt instead of the source speaker's own voice.
Everything upstream and downstream is identical.

**Order of operations**

| Step | Cell | Note |
|---|---|---|
| 1 | 1–7 | normal setup, models loaded |
| 2 | 3b | prepare + listen to the generic voice |
| 3 | 8b | freeze transcripts from the completed cloned run |
| 4 | 9 | make sure input videos are staged |
| 5 | 8c | run the baseline |
| 6 | eval notebook | `RUN_SUFFIX = "_nocloning"`, then the comparison cell |

**What is isolated.** `no_cloning` swaps the prompt inside
`synthesise_segment` and nothing else. Speaker references are still built,
so SpeakerSim in the baseline is still measured against the *source
speaker* — which is the point. The baseline's SpeakerSim is the floor that
makes the cloned run's 0.86 mean something.

**What to expect.** SpeakerSim should fall substantially (a generic voice
has no reason to resemble a Hindi news anchor). WER_tts should be flat or
slightly better, since a clean studio prompt is easier to synthesise from
than separated vocals. That pattern is the claim: cloning buys speaker
similarity at no intelligibility cost.

**If WER_tts improves a lot in the baseline**, say so rather than burying
it — it means source-cloning costs some intelligibility, which is a real
finding and a more interesting paper than the one where nothing moves.

**For the XTTS arm.** The same three edits port over: the `no_cloning`
block in the synthesis helper, `_RUN_SUFFIX` on the output paths, and the
freeze load. Use the *same* `generic_en.wav` and the *same* freeze cache —
if the two backends get different generic voices or different transcripts,
the four-way table is not readable.


In [ ]:
# ─────────────────────────────────────────
# CELL 8b — BUILD THE FREEZE CACHE FROM THE COMPLETED CLONED RUN
# ─────────────────────────────────────────
# RUN THIS ONCE, BEFORE THE BASELINE. It reads the eval records you already
# have and writes frozen_segments/<clip>.json. The baseline run then reuses
# those exact segment boundaries, speaker labels, source text and English MT
# text instead of re-decoding them.
#
# Why this matters more than it looks: WER_tts is dub_hyp vs english_mt_text.
# If the baseline re-runs NLLB and gets even slightly different English, the
# two conditions are being scored against two different references and the
# WER difference you report is partly a translation difference. Freezing
# removes that ambiguity entirely, and it costs one cell.
import glob, json, os
from pathlib import Path

SRC_RECORDS = f"{OUTPUT_DIR}/eval_records"          # the cloned run
FRZ_DIR     = Path(OUTPUT_DIR) / CFG.get("freeze_dir", "frozen_segments")
FRZ_DIR.mkdir(parents=True, exist_ok=True)

OVERWRITE = False        # True to rebuild an existing cache

recs = sorted(glob.glob(f"{SRC_RECORDS}/*_eval.json"))
print(f"{len(recs)} cloned-run records in {SRC_RECORDS}\n")

made = skipped = bad = 0
for p in recs:
    r    = json.load(open(p, encoding="utf-8"))
    cid  = r["clip_id"]
    out  = FRZ_DIR / f"{cid}.json"
    if out.exists() and not OVERWRITE:
        skipped += 1; continue

    segs = r.get("segments", [])
    if not segs:
        print(f"  [bad]  {cid}: record has no per-segment list — cannot freeze")
        bad += 1; continue
    if not any((s.get("text_en_mt") or "").strip() for s in segs):
        print(f"  [bad]  {cid}: no English MT text in segments")
        bad += 1; continue

    json.dump({
        "clip_id":  cid,
        "det_lang": r.get("src_language", "hi"),
        "src_code": r.get("nllb_src_code", "hin_Deva"),
        "segments": [{
            "start":   round(float(s.get("start", 0.0)), 3),
            "end":     round(float(s.get("end", 0.0)), 3),
            "speaker": s.get("speaker") or "SPEAKER_00",
            "text":    s.get("text_src", ""),
            "text_en": s.get("text_en_mt", ""),
        } for s in segs],
    }, open(out, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    made += 1
    print(f"  [ok]   {cid}: {len(segs)} segments")

print(f"\nfroze {made}, skipped {skipped} (already present), bad {bad}")
print(f"cache: {FRZ_DIR}")
if bad:
    print("Clips that failed to freeze will re-decode ASR/MT in the baseline. "
          "Either exclude them from the paired comparison or re-run the cloned "
          "arm for them first.")


In [ ]:
# ─────────────────────────────────────────
# CELL 8c — RUN THE NO-CLONING BASELINE  (CHANGE 3)
# ─────────────────────────────────────────
# Prerequisites, in order:
#   CELL 1-7   normal setup, models loaded
#   CELL 3b    generic voice prepared
#   CELL 8b    freeze cache built from the cloned run
#   CELL 9     input videos present in VID_DIR
#
# Writes to *_nocloning paths only. Nothing from the cloned run is touched.
# Resumable: finished clips are skipped, so a Colab disconnect costs one clip.
import glob, os, traceback, json
from pathlib import Path

BASELINE_SUFFIX = "_nocloning"
VID_DIR   = globals().get("VID_DIR", "/content/batch_input/videos")
VIDEO_EXT = (".mp4", ".mkv", ".mov", ".webm", ".avi")

vids = sorted(p for p in glob.glob(f"{VID_DIR}/*") if p.lower().endswith(VIDEO_EXT))
assert vids, f"No videos in {VID_DIR} — run CELL 9 first."

# ── preflight: fail loudly here rather than 40 minutes in ──
_gv = CFG.get("generic_voice_path")
assert _gv and os.path.exists(_gv), "Run CELL 3b — generic voice not prepared."
_frz_dir = Path(OUTPUT_DIR) / CFG.get("freeze_dir", "frozen_segments")
_n_frozen = len(list(_frz_dir.glob("*.json"))) if _frz_dir.exists() else 0
print(f"generic voice : {_gv}")
print(f"frozen clips  : {_n_frozen}")
print(f"videos        : {len(vids)}")
if _n_frozen < len(vids):
    print("⚠ fewer frozen caches than videos. Unfrozen clips will re-decode "
          "ASR/MT and will NOT be a clean paired comparison. Run CELL 8b.")

_saved = {k: CFG.get(k) for k in ("no_cloning", "run_tag")}
CFG["no_cloning"] = True
CFG["run_tag"]    = BASELINE_SUFFIX

out_dir = Path(OUTPUT_DIR)
ok, fail, skip = [], [], []
try:
    for i, v in enumerate(vids, 1):
        stem = Path(v).stem
        rec  = out_dir / f"eval_records{BASELINE_SUFFIX}" / f"{stem}_eval.json"
        if rec.exists():
            print(f"\n[{i}/{len(vids)}] {stem} — already done, skipping")
            skip.append(stem); continue
        print(f"\n{'='*70}\n[{i}/{len(vids)}] BASELINE {stem}\n{'='*70}")
        try:
            dub_video(v)
            ok.append(stem)
        except Exception as e:
            print(f"!! {stem} FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()
            fail.append(stem)
finally:
    CFG.update(_saved)          # always restore, even on Ctrl-C
    print(f"\nCFG restored: no_cloning={CFG['no_cloning']}, "
          f"run_tag={CFG['run_tag']!r}")

print(f"\nok {len(ok)} | skipped {len(skip)} | failed {len(fail)}")
if fail:
    print("failed:", fail)
print(f"\nBaseline videos : {OUTPUT_DIR}/dubbed{BASELINE_SUFFIX}")
print(f"Baseline records: {OUTPUT_DIR}/eval_records{BASELINE_SUFFIX}")
print("Next: run CLF5_eval_standalone_baseline.ipynb with RUN_SUFFIX "
      f'= "{BASELINE_SUFFIX}" to get dub_hyp / SpeakerSim / metrics, then the '
      "comparison cell.")
